# New Use Case Work Flow
###Stage 1 — One-time foundation (already done — skip)
* Steps 1.1 through 1.4.2 and 1.5 created the runs table, extracted the Codebook (2,133 pages), built the candidates → review → bronze-match chain, and created the approval tables. 
* Only re-run these if a new Codebook version arrives — then re-execute 1.2 with the new document (new codebook_version), followed by 1.3 → 1.4.2 for that version.

### Stage 2 — Define the variable dictionary
1. Edit Step 1.5.1: add one entry per variable family (var_name_root, var_wave_range, var_stem, var_source_page, var_label).
2. Verify each label and page against hrs_document_pages (as we did for HEIGHT/WEIGHT/RACEM), and confirm the expanded names exist as Bronze columns.
#### Tip: 
the approved_variable_dictionary_a suffix is per-use-case — for use case B, add a _b dictionary (or a sibling config notebook) rather than mixing use cases.

### Stage 3 — Approve the variables
* In Step 1.5.2, update the config cell's use_case_name (it's currently hardcoded to the BMI use case) and point the %run at the right dictionary.
* Run top-to-bottom. Checkpoints: unique variables = expected count, Exact Bronze matches = all, MERGE OK, verify table shows every row Approved / Pending with bronze_column_match = true.

### Stage 4 — Compute cohort coverage
Update use_case_name in Step 1.5.3's config cell, run it, and review the summary: HAS_DATA/NO_DATA should be sensible (cohorts that entered the study later will show NO_DATA in early waves).

### Stage 5 — Human review sign-off
The notebooks leave reviewed_by/reviewed_at NULL and wave_coverage_status = 'Pending' by design. Record your review decisions (a simple UPDATE against the approved tables) so the metadata carries your sign-off.

### Stage 6 — Run the Agent
Launch the agent (src/agent in hrs_dbx_agent_repo) against the new use case. It reads only the metadata you just built — approved variables, cohort coverage, Codebook pages — then:

* generate_warehouse_artifacts routes the variables against the live UC information schema: variables mapping to existing CDM homes produce a coverage report (no drafts); true gaps produce tiered DDL/DML drafts.
* generate_codebook_label_dimension if the use case adds categorical variables — it emits a human-readable cross-check vs dim_cohort, never a new reference table.
* display_draft_artifacts presents everything as DRAFT with a review checklist — the agent never executes anything.

### Stage 7 — Review, deploy, verify (human executes)
* Apply accepted drafts following your type standards (DOUBLE for measures, TINYINT for codes, STRING for hhid/pn/wave_year) and the 5-entity CDM.
* Commit the notebooks and warehouse scripts to git in hrs_dbx_repo; execute the DDL/DML.
* Re-run the agent's coverage report — the new variables should now resolve to their CDM homes — then query the loaded tables.